In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdf(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    print(f"Found {len(pdf_files)} PDF files to process")
    for pdf in pdf_files:
        print(f"\n Processing: {pdf.name}")
        try:
            loader = PyMuPDFLoader(str(pdf))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf.name
                doc.metadata['file_type'] = pdf.suffix
            all_documents.extend(documents)
            print(f"Loaded {len(all_documents)} pages")
        except Exception as e:
            print(e)
        print("\n Total documents loaded: ", len(all_documents))
        return all_documents
all_pdf_documents = process_all_pdf("../data")


In [ ]:
all_pdf_documents

In [ ]:
## Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print("\nExample chunk: ")
        print(f"Content {split_docs[0].page_content[:200]}...")
        print(f"Metadata :{split_docs[0].metadata}")
    return split_docs

In [ ]:
chunks  = split_documents(all_pdf_documents)


Embedding and vectorStoreDB


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
import os
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embeddings generation using sentence-transformers"""
    def __init__(self,model_name:str="all-MiniLM-L6-V2"):
        """
        Initializes Embedding Manager
        Args:
            model_name:HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads Sentence Embedding Model"""
        try:
            print("Loading Sentence Embedding Model..."+self.model_name)
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully.Embedding dimensions :{self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading Sentence Embedding Model: {e}")
            raise

    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """Generate embeddings for list of texts"""
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated Embeddings with shapes : {len(embeddings)}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Vector Store DB


In [ ]:
class VectorStore:
    """Manages document embeddings in chromadb vector store"""
    def __init__(self,collection_name:str="collection",persist_directory:str="../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client=None
        self.collection =None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromadb client and collection"""
        try:
            # create persistent chromadb client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name,metadata={"description":"PDF document embeddings for RAG","hnsw:space": "cosine"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error initializing Vector store : {e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """Add documents and their embeddings to vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings do not match")
        print(f"Adding {len(documents)} documents and {len(embeddings)} embeddings to vector store")

        #Prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )

            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise
        print(f"Successfully added {len(documents)} documents to vector store")
        print(f"Total documents in collection: {self.collection.count()}")
vectorstore = VectorStore()
vectorstore

In [ ]:
# convert text to embeddings
texts = [doc.page_content for doc  in chunks]
# generate embeddings
embeddings  = embedding_manager.generate_embeddings(texts)
# store in vector db
vectorstore.add_documents(chunks,embeddings)


Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query based retrieval from vector store"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.5)->List[Dict[str, Any]]:
        """Retrieve relevant documents for a query and returns list of dic containing retrieved doc and metadata"""
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]


                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score = 1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "metadata": metadata,
                            "content": document,
                            'similarity_score': similarity_score,
                            "distance": distance,
                            "rank": i+1,
                        })
                print(f"Retrieved {len(retrieved_docs)} retrieved documents after filtering")

            else:
                print("No retrieved documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents for query: {query}")
            print(f"Actual error:{e}")
            return []

rag_retriever = RAGRetriever(vector_store=vectorstore,embedding_manager=embedding_manager)

In [ ]:
rag_retriever.retrieve("what are basic linux commands")


Integration Vectordb Context pipeline with LLM output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)


def rag_simple(query, retriever, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found"

    # Generate answer using Groq LLM
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke([prompt])

    return response.content

In [ ]:
answer = rag_simple("what to do in lab3 for linux ",rag_retriever)
print(answer)

Enhanced RAG Pipeline Features

In [ ]:
def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):
    results = retriever.retrieve(query, top_k=top_k,score_threshold=min_score)
    if not results:
        return {'answer':"No relevant context found.",'sources':[],'confidence':0.0,'context':""}

    ## prepare context and sources

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""
    sources = [{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source',"unknown")),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:120]+'...'
    } for doc in results]
    confidance = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"Use the following context to answer the question concisely.\n Context:\n{context}\n\nQuestion:\n{query}"
    response = llm.invoke([prompt.format(context=context,query=query)])

    output = {
        'answer':response.content,
        'sources':sources,
        'confidence':confidance,
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("What are basic linux commands",rag_retriever,llm,top_k=3,min_score=0.1,return_context=True)
print("Answer:",result['answer'])
print("Sources:",result['sources'])
print("Confidence:",result['confidence'])
print("Context Preview:",result['context'][:300])

Advanced RAG Streaming Citations History Summarization

In [ ]:
import time
from typing import Dict

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}"""
            if stream:
                for i in range(0, len(prompt), 80):
                    #print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

            # Add citations to answer
            citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
            answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

            # Optionally summarize answer
            summary = None
            if summarize and answer:
                summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

            # Store query history
            self.history.append({
                'question': question,
                'answer': answer,
                'sources': sources,
                'summary': summary
            })

            return {
                'question': question,
                'answer': answer_with_citations,
                'sources': sources,
                'summary': summary,
                'history': self.history
            }


# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What are basic commands in linux?", top_k=3, min_score=0.3, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])